# Решения: score, сигмоида и порог

**Для преподавателя.** Полный эталон к `lesson.ipynb` и `homework.ipynb`; ученикам до сдачи не показывать.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def find_bank_csv() -> Path:
    for path in (Path("bank_marketing_slim.csv"), Path("../../data/bank_marketing_slim.csv")):
        if path.exists():
            return path.resolve()
    return "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_10_churn_logreg/data/bank_marketing_slim.csv"


CSV_PATH = find_bank_csv()
df = pd.read_csv(CSV_PATH)
target = df["y"].eq("yes").astype(int)
assert len(df) > 0 and set(target.unique()) == {0, 1}
assert "duration" in df.columns  # колонка видна только для разбора утечки
print(f"Строк: {len(df)}; доля yes: {target.mean():.3f}")


## Урок. 1. Момент предсказания

In [ ]:
before_call = [c for c in df.columns if c not in {"y", "duration"}]
after_call = ["duration"]
assert "duration" not in before_call


## Урок. 2. Объяснение leakage

In [ ]:
LEAKAGE_NOTE = ("duration измеряет длительность уже состоявшегося звонка и становится известна только после контакта. "
"В момент выбора клиента для будущего обзвона такого значения нет. Модель увидит связь с результатом, "
"но этот сигнал нельзя воспроизвести в реальном процессе; test-оценка будет вводить в заблуждение.")
assert len(LEAKAGE_NOTE) >= 180


## Урок. 3. Линейный score

In [ ]:
def linear_score(value, weight, intercept):
    return intercept + weight * value

assert linear_score(2.0, 1.5, -1.0) == 2.0


## Урок. 4–5. Сигмоида и монотонность

In [ ]:
def sigmoid(z):
    arr = np.asarray(z, dtype=float)
    return 1.0 / (1.0 + np.exp(-arr))

score_grid = np.arange(-6.0, 7.0, 1.0)
probability_grid = sigmoid(score_grid)
is_increasing = bool(np.all(np.diff(probability_grid) > 0))
assert is_increasing and abs(float(sigmoid(0.0)) - 0.5) < 1e-9


## Урок. 6. Порог

In [ ]:
def apply_threshold(proba, threshold=0.5):
    return (np.asarray(proba) >= threshold).astype(int)

demo = np.array([0.10, 0.30, 0.49, 0.50, 0.82])
assert apply_threshold(demo, 0.5).tolist() == [0, 0, 0, 1, 1]


## Урок. 7. Объём обзвона

In [ ]:
demo_proba = np.array([0.06, 0.18, 0.24, 0.31, 0.47, 0.52, 0.68, 0.79, 0.91])
thresholds = [0.2, 0.35, 0.5, 0.65, 0.8]
selected_counts = [int(apply_threshold(demo_proba, t).sum()) for t in thresholds]
assert selected_counts == sorted(selected_counts, reverse=True)


## Урок. 8. Порог под бюджет

In [ ]:
budget = 3
threshold_grid = np.arange(0.05, 1.0, 0.05)
budget_threshold = next(t for t in threshold_grid if apply_threshold(demo_proba, t).sum() <= budget)
chosen = np.flatnonzero(apply_threshold(demo_proba, budget_threshold))
assert len(chosen) <= budget


## Урок. 9. Контракт решения

In [ ]:
def campaign_decision(proba, threshold):
    labels = apply_threshold(proba, threshold)
    return labels, int(labels.sum())

labels, count = campaign_decision(demo_proba, 0.5)
assert count == 4


## ДЗ. A1–A2. Score и решение

In [ ]:
ages = np.array([22, 35, 47, 61], dtype=float)
scores = -3 + 0.06 * ages
age_proba = sigmoid(scores)
age_pred = apply_threshold(age_proba, 0.5)
assert age_proba.shape == ages.shape


## ДЗ. A3. Безопасные признаки

In [ ]:
FEATURE_COLUMNS = [column for column in df.columns if column not in {"y", "duration"}]
assert "duration" not in FEATURE_COLUMNS, "LEAKAGE: duration известна только после звонка"
assert "y" not in FEATURE_COLUMNS
assert "duration" not in FEATURE_COLUMNS


## ДЗ. Challenge. Контрпример

In [ ]:
COUNTEREXAMPLE = ("Даже если duration резко повышает test-метрику, результат непригоден для кампании: в момент решения, "
"кому звонить, duration ещё не существует. Она появляется после звонка и частично отражает сам отклик. "
"Следовательно, offline test проверяет задачу с недоступным сигналом, а не будущий процесс. Сравнивать модели нужно только на признаках, доступных до звонка.")
assert len(COUNTEREXAMPLE) >= 240


## ДЗ. Challenge. Устойчивость бюджета

In [ ]:
budgets = [2, 3, 4]
budget_thresholds = [next(t for t in threshold_grid if apply_threshold(demo_proba, t).sum() <= b) for b in budgets]
assert budget_thresholds == sorted(budget_thresholds, reverse=True)
